# Pre-procesamiento para entrenamiento de modelo de predicción de categorías


## Importación de librerias necesarias


In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import sys

def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path

src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")
print(f"TFHUB_CACHE_DIR: {os.environ.get('TFHUB_CACHE_DIR')}")


JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
TFHUB_CACHE_DIR: /mnt/d/Maestría/Amazon Reviews Code/tf_cache


In [3]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T, DataFrame
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pyspark.ml.functions import array_to_vector, vector_to_array
from pyspark.ml.feature import VectorAssembler

In [4]:
from src.utils.spark import SparkUtils
spark_utils = SparkUtils('premodeling_predict_category_model')
spark = spark_utils.spark


2025-10-19 19:45:26.466024: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-19 19:45:26.477562: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-19 19:45:26.481099: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-19 19:45:26.490346: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-19 19:45:28.662744: W tensorflow/compiler/tf2

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-317e978c-8d25-4ee0-9fc1-06ddd50a6fa6;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 122ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

## Cargar datos fuente de entrenamiento

In [5]:
GOLD_ENCODING = 'gold.encoding'
GOLD_PREMODELING = 'gold.premodeling'
GOLD_SCHEMA_CLUSTER = 'gold.cluster'
SCHEMA = 'silver.preprocess'

In [6]:
sample_equitative_hierarchical = spark.read.format('delta').load(spark_utils.path(
    'sample_equitative_hierarchical', catalog=GOLD_SCHEMA_CLUSTER
))

main_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = GOLD_PREMODELING
))

reviews_indexed = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed', catalog = SCHEMA
))

reviews_embeddings_equi = spark.read.format('delta').load(spark_utils.path(
    'reviews_embeddings_equi', catalog = GOLD_ENCODING
))

In [7]:
df_vectorized_pca = spark.read.format('delta').load(spark_utils.path(
    'df_vectorized_pca', catalog = GOLD_PREMODELING
))

df_reviews_vectorized_pca = spark.read.format('delta').load(spark_utils.path(
    'df_reviews_vectorized_pca', catalog = GOLD_PREMODELING
))

In [8]:
REGENERATE_INTERMEDIATE_TABLES = False

## Construir datasets para entrenamiento

### Construir datasets de productos y reseñas para predicción de calificación

In [9]:
tmp_products_training_data = (
    sample_equitative_hierarchical.alias('A').join(
        main_category_encoded.alias('B'),
        on = 'parent_asin',
        how = 'inner'
    )
    .select(
        'A.parent_asin',
        'A.cluster_hierarchical',
        'A.features',
        *[
            F.col(f'B.{col}').alias(col)
            for col in main_category_encoded.columns
            if col.startswith('main_category_')
        ]
    )
)

In [10]:
tmp_reviews_training_data = (
    reviews_embeddings_equi.alias('A').join(
        reviews_indexed.alias('B'),
        on = 'review_id',
        how = 'inner'
    ).join(
        sample_equitative_hierarchical.alias('C'),
        on = 'parent_asin',
        how = 'inner'
    ).select(
        'B.rating',
        'B.rating_boolean',
        'B.helpful_vote',
        'A.text_embeddings',
        'C.cluster_hierarchical'
    )
)

In [11]:
final_training_data_rating_numeric = (
    tmp_reviews_training_data.alias('A').join(
        tmp_products_training_data.alias('B'),
        on = 'cluster_hierarchical',
        how = 'inner'
    ).select(
        'A.rating', 
        'A.rating_boolean',
        'A.helpful_vote',
        F.concat(
            F.col('A.text_embeddings'),
            vector_to_array(F.col('B.features'))
        ).alias('combined_features')
    )
)

In [12]:
sample_row = final_training_data_rating_numeric.limit(1).collect()[0]
combined_features_size = len(sample_row['combined_features'])

select_exprs = []

for i in range(combined_features_size):
    select_exprs.append(F.col("combined_features")[i].alias(f"combined_feat_{i}"))

final_training_data_rating_array = final_training_data_rating_numeric.select(
    F.col("rating").cast("float").alias("rating"),
    F.col("helpful_vote").cast("int").alias("helpful_vote"),
    *select_exprs
)

final_training_data_rating_array_boolean = final_training_data_rating_numeric.select(
    F.col("rating_boolean").cast("float").alias("rating_boolean"),
    F.col("helpful_vote").cast("int").alias("helpful_vote"),
    *select_exprs
)

final_training_data_rating_array_regression = final_training_data_rating_numeric.select(
    (F.col("rating").cast("float") / 5).alias("rating_regression"),
    F.col("helpful_vote").cast("int").alias("helpful_vote"),
    *select_exprs
)

In [16]:
if True:
    (
        final_training_data_rating_array
            .limit(100_000)
            .coalesce(1)
            .write
            .mode("overwrite")
            .option('overwriteSchema', 'true')
            .parquet(spark_utils.path(
                'final_training_data_rating_array', catalog = GOLD_PREMODELING
            ))
    )

In [17]:
if True:
    (
        final_training_data_rating_array_boolean
            .limit(100_000)
            .coalesce(1)
            .write
            .mode("overwrite")
            .option('overwriteSchema', 'true')
            .parquet(spark_utils.path(
                'final_training_data_rating_array_boolean', catalog = GOLD_PREMODELING
            ))
    )

In [18]:
if True:
    (
        final_training_data_rating_array_regression
            .limit(100_000)
            .coalesce(1)
            .write
            .mode("overwrite")
            .option('overwriteSchema', 'true')
            .parquet(spark_utils.path(
                'final_training_data_rating_array_regression', catalog = GOLD_PREMODELING
            ))
    )

### Construir datasets de productos para predicción de categorías

In [ ]:
category_prediction_products_training_data = (
    sample_equitative_hierarchical.alias('A').join(
        main_category_encoded.alias('B'),
        on = 'parent_asin',
        how = 'inner'
    )
    .select(
        'A.features',
        *[
            F.col(f'B.{col}').alias(col)
            for col in main_category_encoded.columns
            if col.startswith('main_category_')
        ]
    )
)

In [ ]:
sample_row = category_prediction_products_training_data.limit(1).collect()[0]
features_size = len(sample_row['features'])

category_prediction_products_training_data_array = category_prediction_products_training_data.select(
    *[
        F.col(f'{col}').alias(col)
        for col in category_prediction_products_training_data.columns
        if col.startswith('main_category_')
    ],
    vector_to_array(F.col('features')).alias('features_array')
)

select_exprs = []

for i in range(features_size):
    select_exprs.append(F.col("features_array")[i].alias(f"feat_{i}"))

final_training_data_category_prediction = category_prediction_products_training_data_array.select(
    *[
        F.col(f'{col}').alias(col)
        for col in category_prediction_products_training_data_array.columns
        if col.startswith('main_category_')
    ],
    *select_exprs
)

In [ ]:
if True:
    (
        final_training_data_category_prediction
            .coalesce(1)
            .write
            .mode("overwrite")
            .option('overwriteSchema', 'true')
            .parquet(spark_utils.path(
                'final_training_data_category_prediction', catalog = GOLD_PREMODELING
            ))
    )